# OSC Pick-and-Place Training (Colab)

Phase 3, built directly on `5-arm_project_osc`'s Phase 2 (Grasp) success. Runs `train_osc_pick_place_bc_parallel.py` from the `Arm-OSC-Pick-and-Place` repo on a Colab runtime — a behavior-cloning-warm-started SAC training, applied from the start rather than trying pure RL first (see `IMP_NOTES.md`: pure RL never found a single grasp success in the grasp-only task across 585k/1M steps, and this task is a longer, harder multi-stage sequence). Cell 5.5 collects real successful pick-and-place demonstrations via a hand-scripted routine first; cell 6 seeds the replay buffer and pretrains the actor from them before RL fine-tuning begins.

**Runtime type**: a plain CPU runtime is fine — no need to select a GPU. This workload is CPU-bound MuJoCo physics plus a tiny MLP; `device="cpu"` is set explicitly in the training script regardless, and a T4 wouldn't speed this up.

**Why Drive is mounted**: Colab's local disk is wiped whenever the runtime disconnects or recycles (idle timeout, 12h session cap, etc.) — a multi-hour training run WILL eventually hit this. Checkpoints are symlinked into Google Drive below so they survive a disconnect; only re-run cells 1-4 to resume watching a run, and cells 5.5-6 again to re-collect demonstrations and continue/restart training.

In [1]:
# Cell 1 — mount Google Drive (checkpoints and the final model save both
# land here, not on Colab's ephemeral local disk)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2 — clone the repo. Leave the token prompt blank if the repo is public;
# paste a GitHub Personal Access Token (repo scope) if it's private.
import getpass, os

REPO_URL = "https://github.com/kaustubhadhe1206/Arm-OSC-Pick-and-Place.git"
REPO_DIR = "Arm-OSC-Pick-and-Place"

token = getpass.getpass("GitHub token (leave blank if repo is public): ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {clone_url}

%cd {REPO_DIR}

Cloning into 'Arm-OSC-Pick-and-Place'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 88 (delta 2), reused 88 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 4.83 MiB | 9.17 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Arm-OSC-Pick-and-Place


In [3]:
# Cell 3 — point checkpoints at Drive via a symlink, so the training script's
# existing `save_path="./checkpoints/"` transparently writes to Drive instead
# of Colab's local (ephemeral) disk, with no changes needed to the script
# itself.
import os

DRIVE_CKPT_DIR = "/content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

if os.path.islink("checkpoints") or os.path.isdir("checkpoints"):
    !rm -rf checkpoints
!ln -s {DRIVE_CKPT_DIR} checkpoints

print("Checkpoints will be saved to:", DRIVE_CKPT_DIR)

Checkpoints will be saved to: /content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints


In [4]:
# Cell 4 — install dependencies. torch is already preinstalled on Colab (CUDA
# build) — that's fine, the training script forces device="cpu" regardless.
!pip install -q mujoco gymnasium stable-baselines3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 53.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 16.0 MB/s eta 0:00:00


In [5]:
# Cell 5 — sanity check: how many CPU cores does this runtime actually have?
# (5-arm_project_osc's Colab sessions showed 2 vCPUs on the free tier.)
import os
print("CPU count:", os.cpu_count())

CPU count: 2


In [6]:
# Cell 5.5 — collect pick-and-place demonstrations via a hand-scripted (no
# RL) routine, used to warm-start training below. This task's scripted
# routine succeeds ~75-85% of the time (see IMP_NOTES.md) -- somewhat
# slower per-attempt than the grasp-only task's demo collection since each
# successful episode is a full approach-grasp-lift-carry-release-settle
# sequence (~250-350 transitions vs ~150-200 for grasp-only). Only needs to
# run once per Colab session; demonstrations.npz persists in this session's
# local disk for the rest of it (re-run if the runtime disconnects and you
# start a fresh session).
!python collect_demonstrations.py 300

  attempts=20 successes=14/300 (success rate so far: 70%)
  attempts=40 successes=27/300 (success rate so far: 68%)
  attempts=60 successes=40/300 (success rate so far: 67%)
  attempts=80 successes=57/300 (success rate so far: 71%)
  attempts=100 successes=73/300 (success rate so far: 73%)
  attempts=120 successes=88/300 (success rate so far: 73%)
  attempts=140 successes=101/300 (success rate so far: 72%)
  attempts=160 successes=116/300 (success rate so far: 72%)
  attempts=180 successes=132/300 (success rate so far: 73%)
  attempts=200 successes=145/300 (success rate so far: 72%)
  attempts=220 successes=162/300 (success rate so far: 74%)
  attempts=240 successes=176/300 (success rate so far: 73%)
  attempts=260 successes=191/300 (success rate so far: 73%)
  attempts=280 successes=206/300 (success rate so far: 74%)
  attempts=300 successes=223/300 (success rate so far: 74%)
  attempts=320 successes=239/300 (success rate so far: 75%)
  attempts=340 successes=255/300 (success rate so 

In [7]:
# Cell 6 — run BC-warm-started training. This streams SB3's logging table
# live and blocks until 1,000,000 steps complete or the runtime
# disconnects — checkpoints every 12,500 steps land in Drive via the
# symlink either way, so a disconnect loses at most that much progress.
# NOTE: this task is longer/harder than the grasp-only one that needed the
# full 1M steps to fully converge -- watch ep_len_mean/ep_rew_mean near the
# end and consider extending total_timesteps in the script if it's still
# clearly improving rather than assuming 1M is automatically enough.
!python train_osc_pick_place_bc_parallel.py

2026-09-10 01:15:41.948442: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-10 01:15:42.024800: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Loaded 86179 demonstration transitions from /content/Arm-OSC-Pick-and-Place/demonstrations.npz
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other 

In [ ]:
# Cell 7 — only relevant if cell 6 finished without disconnecting: the FINAL
# model.save() writes to the repo directory (Colab's local disk), not
# checkpoints/ — copy it to Drive too so it isn't lost.
!cp sac_franka_osc_pick_place_bc_parallel.zip /content/drive/MyDrive/Arm-OSC-Pick-and-Place-checkpoints/ 2>/dev/null || echo "Not found yet — training may not have completed."